# Week 5 — Logic & Automated Planning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aofphy/SCI193611_ARTIFICIAL_INTELLIGENCE/blob/main/labs/w05_logic_planning.ipynb)

**Objective:** แทนความรู้ด้วยตรรกะเชิงประพจน์ และแก้ปัญหาด้วยการวางแผนอัตโนมัติ (STRIPS).

เนื้อหานี้รันได้ทันทีด้วย Python มาตรฐาน (ไม่ต้องติดตั้งไลบรารีเพิ่ม).


## 1) Propositional Logic — DPLL
ตัวแก้ปัญหา satisfiability แบบ backtracking. แต่ละ clause เป็น list ของ literal (เลขบวก = ตัวแปรจริง, เลขลบ = นิเสธ).


In [ ]:
# Propositional logic: DPLL satisfiability solver
def dpll(clauses, symbols, model=None):
    model = model or {}
    def clause_val(c):
        if any((model.get(abs(l)) is True and l>0) or (model.get(abs(l)) is False and l<0) for l in c):
            return True
        if all(model.get(abs(l)) is not None for l in c):
            return False
        return None
    vs=[clause_val(c) for c in clauses]
    if any(v is False for v in vs): return None
    if all(v is True for v in vs):  return dict(model)
    p=[s for s in symbols if s not in model][0]
    for val in (True, False):
        m=dict(model); m[p]=val
        r=dpll(clauses, symbols, m)
        if r is not None: return r
    return None

# (A or B) and (not A or C) and (not C)   [1=A, 2=B, 3=C; negative = NOT]
clauses=[[1,2],[-1,3],[-3]]
print("SAT model:", dpll(clauses,[1,2,3]))   # -> A=False, B=True, C=False

## 2) Automated Planning — STRIPS + BFS
ค้นหาลำดับการกระทำที่พาจากสถานะเริ่มต้นไปถึงเป้าหมาย.


In [ ]:
# STRIPS planning by breadth-first search over states
from collections import deque

def plan(init, goal, actions):
    init=frozenset(init); goal=set(goal)
    seen={init}; q=deque([(init,[])])
    while q:
        s,path=q.popleft()
        if goal<=s: return path
        for name,pre,add,dele in actions:        # action = (name, preconds, add-list, delete-list)
            if set(pre)<=s:
                ns=frozenset((s-set(dele))|set(add))
                if ns not in seen:
                    seen.add(ns); q.append((ns,path+[name]))
    return None

actions=[("move_a_to_b", ["a_clear","b_clear","a_on_table"], ["a_on_b"], ["b_clear","a_on_table"])]
print("Plan:", plan({"a_clear","b_clear","a_on_table"}, {"a_on_b"}, actions))

## 3) TODO (ฝึกต่อ)
- เพิ่มปัญหา blocks-world ที่ซับซ้อนขึ้น (3+ บล็อก) และเปรียบเทียบความยาวแผน
- แทน BFS ด้วย A* + heuristic (จำนวน goal ที่ยังไม่สำเร็จ)
- เพิ่ม unit-propagation ใน DPLL เพื่อเร่งความเร็ว
